# Previsão da concentração de CO (Air Quality UCI) — Comparação e Otimização de Modelos de Regressão

## Definição do problema

O objetivo deste notebook é construir e comparar modelos de **regressão** capazes de estimar a concentração de monóxido de carbono no ar a partir de leituras de sensores químicos e variáveis ambientais/temporais.

- **Variável-alvo:** `CO(GT)` — concentração de referência de CO (mg/m³), medida por um analisador de referência certificado.
- **Features:** respostas dos sensores químicos (`PT08.S1(CO)` a `PT08.S5(O3)`), temperatura (`T`), umidade relativa (`RH`), umidade absoluta (`AH`) e as features temporais derivadas `hour`, `weekday` e `month`. As demais variáveis `*(GT)` (concentrações de referência de outros poluentes) **não** são usadas como entrada, pois são medidas pelo mesmo tipo de analisador de referência que gerou o alvo — usá-las causaria vazamento de informação (data leakage).
- **Dataset:** [Air Quality UCI](https://archive.ics.uci.edu/dataset/360/air+quality) — 9357 registros horários coletados por um dispositivo multissensor em uma cidade italiana entre março de 2004 e fevereiro de 2005.

## Escolha dos algoritmos

Diferentemente do notebook de referência (que usa apenas um modelo linear como baseline), esta atividade pede a **comparação e otimização** de dois algoritmos de regressão de famílias diferentes, nenhum deles o `ExtraTreesRegressor`:

1. **`RandomForestRegressor`** — algoritmo da família de **ensembles de árvores (bagging)**, mesma família apresentada no seminário. Um Random Forest treina várias árvores de decisão sobre amostras bootstrap dos dados e sobre subconjuntos aleatórios de features, agregando as previsões (média) para reduzir a variância de uma única árvore. É um dos modelos recomendados pelo [Machine Learning Map do scikit-learn](https://scikit-learn.org/stable/machine_learning_map.html) para problemas de regressão com poucos dados e relações não lineares entre features e alvo — cenário plausível aqui, já que a resposta de sensores químicos a poluentes não é necessariamente linear.

2. **`KNeighborsRegressor`** (dentro de um `Pipeline` com `StandardScaler`) — algoritmo da família de **modelos baseados em instância (instance-based / lazy learning)**, uma família estruturalmente diferente da de ensembles: em vez de aprender parâmetros/regras a partir dos dados de treino, o KNN simplesmente memoriza os exemplos e prevê o valor de um novo ponto pela média dos `k` vizinhos mais próximos no espaço de features. Como o KNN é sensível à escala das variáveis (distância euclidiana), ele é usado dentro de um `Pipeline` com `StandardScaler`, garantindo que sensores com escalas muito diferentes (ex.: `PT08.S1(CO)` na casa das centenas/milhares vs. `AH` menor que 3) contribuam de forma equilibrada para a distância.

Para cada algoritmo, o notebook compara um modelo **baseline** (parâmetros padrão) contra três técnicas de **otimização de hiperparâmetros**: `GridSearchCV`, `RandomizedSearchCV` e otimização bayesiana com **Optuna** (`TPESampler`).

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna

from scipy.stats import randint, uniform

from sklearn.model_selection import train_test_split, KFold, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv("AirQualityUCI.csv", sep=';', decimal=',')

df.head()

In [ ]:
df.info()

## Pré-processamento

In [ ]:
# Remove colunas 'Unnamed' (vazias, geradas pelos ';' finais do CSV) e linhas totalmente vazias
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df = df.dropna(how='all').reset_index(drop=True)

df.shape

In [ ]:
# O dataset usa -200 como código de valor ausente (falha do sensor) -- troca por NaN
df = df.replace(-200, np.nan)

df.isna().mean().sort_values(ascending=False) * 100

In [ ]:
# NMHC(GT) tem ~90% de valores ausentes -- descartada, pois não há como imputar
# uma proporção tão grande de forma confiável
df = df.drop(columns=['NMHC(GT)'])

df.shape

In [ ]:
# Features temporais a partir de Date + Time
dt = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H.%M.%S')

df['hour'] = dt.dt.hour
df['weekday'] = dt.dt.weekday
df['month'] = dt.dt.month

df[['hour', 'weekday', 'month']].describe()

In [ ]:
features = [
    'PT08.S1(CO)', 'PT08.S2(NMHC)', 'PT08.S3(NOx)', 'PT08.S4(NO2)', 'PT08.S5(O3)',
    'T', 'RH', 'AH', 'hour', 'weekday', 'month',
]
target = 'CO(GT)'

# Não usar as outras colunas *(GT) como entrada, para evitar vazamento
data = df[features + [target]].dropna().reset_index(drop=True)

data.shape

In [ ]:
X = data[features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

X_train.shape, X_test.shape

In [ ]:
resultados = []

def avaliar(nome_modelo, tecnica, modelo, X_tr, y_tr, X_te, y_te, tempo_s):
    y_pred = modelo.predict(X_te)
    mae = mean_absolute_error(y_te, y_pred)
    mse = mean_squared_error(y_te, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_te, y_pred)

    resultados.append({
        'Modelo': nome_modelo,
        'Técnica': tecnica,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2': r2,
        'Tempo_s': tempo_s,
    })

    print(f"{nome_modelo} | {tecnica}")
    print(f"MAE:  {mae:.4f}")
    print(f"MSE:  {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²:   {r2:.4f}")
    print(f"Tempo: {tempo_s:.2f}s")

    return y_pred

## Algoritmo 1 — Random Forest Regressor

### Baseline

In [ ]:
inicio = time.time()
rf_base = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)
rf_base.fit(X_train, y_train)
tempo_rf_base = time.time() - inicio

_ = avaliar('RandomForest', 'Baseline', rf_base, X_train, y_train, X_test, y_test, tempo_rf_base)

### Otimização de hiperparâmetros

Hiperparâmetros otimizados (documentação: [`RandomForestRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)):

- **`n_estimators`** — número de árvores da floresta. Mais árvores tendem a estabilizar (reduzir a variância d)a previsão, ao custo de tempo de treino/predição maior; a busca cobre de poucas dezenas a algumas centenas de árvores.
- **`max_depth`** — profundidade máxima de cada árvore. Controla o trade-off viés/variância: árvores rasas tendem a subajustar (alto viés), árvores muito profundas tendem a decorar o treino (alta variância). `None` permite crescer até as folhas ficarem puras, sendo incluído como referência de "sem limite".
- **`min_samples_leaf`** — número mínimo de amostras exigido em uma folha. Valores maiores suavizam o modelo (menos overfitting a ruído pontual dos sensores).
- **`max_features`** — número/fração de features consideradas em cada split. Reduzir esse valor aumenta a diversidade entre as árvores (decorrelaciona os erros), um dos principais mecanismos pelos quais o Random Forest reduz variância em relação a uma única árvore.

A grade do `GridSearchCV` foi limitada a poucos valores por hiperparâmetro (2×3×2×2 = 24 combinações) para manter o custo computacional viável; `RandomizedSearchCV` e Optuna exploram distribuições contínuas/mais amplas sobre esses mesmos hiperparâmetros, com `n_iter`/`n_trials` ≈ 25.

In [ ]:
param_grid_rf = {
    'n_estimators': [100, 300],
    'max_depth': [None, 10, 20],
    'min_samples_leaf': [1, 4],
    'max_features': [1.0, 'sqrt'],
}

inicio = time.time()
grid_rf = GridSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid=param_grid_rf,
    scoring='neg_root_mean_squared_error',
    cv=kfold,
    n_jobs=-1,
)
grid_rf.fit(X_train, y_train)
tempo_grid_rf = time.time() - inicio

print("Melhores parâmetros (Grid):", grid_rf.best_params_)
_ = avaliar('RandomForest', 'GridSearchCV', grid_rf.best_estimator_, X_train, y_train, X_test, y_test, tempo_grid_rf)

In [ ]:
param_dist_rf = {
    'n_estimators': randint(50, 500),
    'max_depth': [None, 5, 10, 15, 20, 30],
    'min_samples_leaf': randint(1, 10),
    'max_features': uniform(0.3, 0.7),
}

inicio = time.time()
random_rf = RandomizedSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_dist_rf,
    n_iter=25,
    scoring='neg_root_mean_squared_error',
    cv=kfold,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
random_rf.fit(X_train, y_train)
tempo_random_rf = time.time() - inicio

print("Melhores parâmetros (Random):", random_rf.best_params_)
_ = avaliar('RandomForest', 'RandomizedSearchCV', random_rf.best_estimator_, X_train, y_train, X_test, y_test, tempo_random_rf)

In [ ]:
def objetivo_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 30),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_float('max_features', 0.3, 1.0),
    }
    modelo = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
    scores = cross_val_score(modelo, X_train, y_train, scoring='neg_root_mean_squared_error', cv=kfold, n_jobs=-1)
    return -scores.mean()

inicio = time.time()
study_rf = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_rf.optimize(objetivo_rf, n_trials=25, show_progress_bar=False)
tempo_optuna_rf = time.time() - inicio

print("Melhores parâmetros (Optuna):", study_rf.best_params)
optuna_rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **study_rf.best_params)
optuna_rf.fit(X_train, y_train)
_ = avaliar('RandomForest', 'Optuna', optuna_rf, X_train, y_train, X_test, y_test, tempo_optuna_rf)

## Algoritmo 2 — K-Nearest Neighbors Regressor (com `StandardScaler`)

### Baseline

In [ ]:
pipe_knn_base = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor()),
])

inicio = time.time()
pipe_knn_base.fit(X_train, y_train)
tempo_knn_base = time.time() - inicio

_ = avaliar('KNN', 'Baseline', pipe_knn_base, X_train, y_train, X_test, y_test, tempo_knn_base)

### Otimização de hiperparâmetros

Hiperparâmetros otimizados (documentação: [`KNeighborsRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html)):

- **`n_neighbors`** (`k`) — número de vizinhos usados na previsão. É o principal controle de viés/variância do KNN: `k` pequeno gera um modelo muito flexível (baixo viés, alta variância, sensível a ruído dos sensores), `k` grande suaviza demais a previsão (alto viés). A busca cobre de poucos vizinhos a algumas dezenas.
- **`weights`** — forma de ponderar os vizinhos (`uniform`: todos com peso igual; `distance`: peso inversamente proporcional à distância). `distance` costuma ajudar quando a densidade dos pontos no espaço de features não é uniforme.
- **`p`** — parâmetro da distância de Minkowski (`p=1` → Manhattan, `p=2` → Euclidiana). Testar as duas permite verificar se a geometria do espaço de features (após padronização) favorece uma métrica específica.

Como o KNN está dentro de um `Pipeline`, os hiperparâmetros são referenciados com o prefixo `knn__` nas buscas.

In [ ]:
param_grid_knn = {
    'knn__n_neighbors': [3, 5, 7, 9, 15, 25],
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [1, 2],
}

inicio = time.time()
grid_knn = GridSearchCV(
    Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsRegressor())]),
    param_grid=param_grid_knn,
    scoring='neg_root_mean_squared_error',
    cv=kfold,
    n_jobs=-1,
)
grid_knn.fit(X_train, y_train)
tempo_grid_knn = time.time() - inicio

print("Melhores parâmetros (Grid):", grid_knn.best_params_)
_ = avaliar('KNN', 'GridSearchCV', grid_knn.best_estimator_, X_train, y_train, X_test, y_test, tempo_grid_knn)

In [ ]:
param_dist_knn = {
    'knn__n_neighbors': randint(2, 50),
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [1, 2],
}

inicio = time.time()
random_knn = RandomizedSearchCV(
    Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsRegressor())]),
    param_distributions=param_dist_knn,
    n_iter=25,
    scoring='neg_root_mean_squared_error',
    cv=kfold,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
random_knn.fit(X_train, y_train)
tempo_random_knn = time.time() - inicio

print("Melhores parâmetros (Random):", random_knn.best_params_)
_ = avaliar('KNN', 'RandomizedSearchCV', random_knn.best_estimator_, X_train, y_train, X_test, y_test, tempo_random_knn)

In [ ]:
def objetivo_knn(trial):
    params = {
        'n_neighbors': trial.suggest_int('n_neighbors', 2, 50),
        'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
        'p': trial.suggest_int('p', 1, 2),
    }
    modelo = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsRegressor(**params)),
    ])
    scores = cross_val_score(modelo, X_train, y_train, scoring='neg_root_mean_squared_error', cv=kfold, n_jobs=-1)
    return -scores.mean()

inicio = time.time()
study_knn = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_knn.optimize(objetivo_knn, n_trials=25, show_progress_bar=False)
tempo_optuna_knn = time.time() - inicio

print("Melhores parâmetros (Optuna):", study_knn.best_params)
optuna_knn = Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsRegressor(**study_knn.best_params))])
optuna_knn.fit(X_train, y_train)
_ = avaliar('KNN', 'Optuna', optuna_knn, X_train, y_train, X_test, y_test, tempo_optuna_knn)

## Comparação entre modelos e técnicas

In [ ]:
df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados[['Modelo', 'Técnica', 'MAE', 'MSE', 'RMSE', 'R2', 'Tempo_s']]
df_resultados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_resultados['Rótulo'] = df_resultados['Modelo'] + ' - ' + df_resultados['Técnica']

axes[0].barh(df_resultados['Rótulo'], df_resultados['RMSE'], color='steelblue')
axes[0].set_xlabel('RMSE (mg/m³)')
axes[0].set_title('RMSE por modelo/técnica (teste)')
axes[0].invert_yaxis()

axes[1].barh(df_resultados['Rótulo'], df_resultados['R2'], color='seagreen')
axes[1].set_xlabel('R²')
axes[1].set_title('R² por modelo/técnica (teste)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
reducao = []
for modelo in df_resultados['Modelo'].unique():
    sub = df_resultados[df_resultados['Modelo'] == modelo].set_index('Técnica')
    rmse_base = sub.loc['Baseline', 'RMSE']
    for tecnica in sub.index:
        if tecnica == 'Baseline':
            continue
        rmse_tec = sub.loc[tecnica, 'RMSE']
        reducao.append({
            'Modelo': modelo,
            'Técnica': tecnica,
            'RMSE_baseline': rmse_base,
            'RMSE_tecnica': rmse_tec,
            'Redução_RMSE_%': (1 - rmse_tec / rmse_base) * 100,
        })

df_reducao = pd.DataFrame(reducao)
df_reducao

In [ ]:
df_tempo_tecnica = df_resultados[df_resultados['Técnica'] != 'Baseline'].groupby('Técnica').agg(
    RMSE_médio=('RMSE', 'mean'),
    Tempo_médio_s=('Tempo_s', 'mean'),
).reset_index()

df_tempo_tecnica

## Conclusão

*(Esta seção será preenchida com base nos números reais obtidos na execução acima — ver células de resultado de `df_resultados`, `df_reducao` e `df_tempo_tecnica`.)*